## Introduction

In this tutorial, we will build a character-level text autocomplete model using a Recurrent Neural Network (RNN) in PyTorch. We will train the model on the text from "warandpeace.txt". This project will help you understand how RNNs can be implemented for text generation tasks and their application in building your own autocomplete model.


## Importing Necessary Libraries

In [1]:
# This is Cell #1

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import random
import re


/Users/<redacted>/Desktop/UMASS/GitHub/assignment-7-rnns-ml-complete-<redacted>/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:295: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/<redacted>/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## Setting Up the Device

In [116]:
# This is Cell #2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Reading and Preprocessing the Data

Now it is time to prepare our training data.


In [76]:
# This is Cell #3

def read_file(filename):
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read().lower()
        # Keep only lowercase letters and standard punctuation (.,!?;:()[])
        text = re.sub(r'[^a-z.,!?;:()\[\] ]+', '', text)
    return text

sequence = read_file("warandpeace.txt")


### Here we will train our model with a simple sequence

We will start by training our model with a simple sequence and repettitive sequence such as `"abcdefghijklmnopqrstuvwxyzabcdef..."`, and we will see if our RNN is capable of learning that pattern or not. This will help you easily verify if your RNN is working correctly or not.

In [29]:
# This is Cell #4

sequence = "abcdefghijklmnopqrstuvwxyz" * 100

## Create Character Mappings

Creating character mappings is essential because RNNs require numerical input to process data. By mapping each unique character to an index and creating a reverse mapping, we convert text data into numerical sequences that the model can understand. This step allows us to encode input text for training and decode the model's output back into readable characters during text generation.



In [ ]:
# This is Cell #5

# Create a list of unique characters from the text sequence
#vocab = sorted(set(sequence + "0123456789"))
vocab = sorted(set(sequence))

# Create two dictionaries for character-index mappings that map each character in vocab to a unique index and vice versa
char_to_idx = {char: idx for idx, char in enumerate(vocab)} 
idx_to_char = {idx: char for idx, char in enumerate(vocab)}

# Convert the entire text based data into numerical data
data = [char_to_idx[char] for char in sequence]

# Print outputs to Verify
print("Vocabulary:", vocab)
print("Character to Index Mapping:", char_to_idx)
print("Index to Character Mapping:", idx_to_char)
print("Numerical Data (First 100):", data[:100]) 

Vocabulary: [' ', '!', '(', ')', ',', '.', ':', ';', '?', '[', ']', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Character to Index Mapping: {' ': 0, '!': 1, '(': 2, ')': 3, ',': 4, '.': 5, ':': 6, ';': 7, '?': 8, '[': 9, ']': 10, 'a': 11, 'b': 12, 'c': 13, 'd': 14, 'e': 15, 'f': 16, 'g': 17, 'h': 18, 'i': 19, 'j': 20, 'k': 21, 'l': 22, 'm': 23, 'n': 24, 'o': 25, 'p': 26, 'q': 27, 'r': 28, 's': 29, 't': 30, 'u': 31, 'v': 32, 'w': 33, 'x': 34, 'y': 35, 'z': 36}
Index to Character Mapping: {0: ' ', 1: '!', 2: '(', 3: ')', 4: ',', 5: '.', 6: ':', 7: ';', 8: '?', 9: '[', 10: ']', 11: 'a', 12: 'b', 13: 'c', 14: 'd', 15: 'e', 16: 'f', 17: 'g', 18: 'h', 19: 'i', 20: 'j', 21: 'k', 22: 'l', 23: 'm', 24: 'n', 25: 'o', 26: 'p', 27: 'q', 28: 'r', 29: 's', 30: 't', 31: 'u', 32: 'v', 33: 'w', 34: 'x', 35: 'y', 36: 'z'}
Numerical Data (First 100): [30, 18, 15, 0, 26, 28, 25, 20, 15, 13, 30, 0, 17, 31, 30, 15, 24, 12,

## Defining the CharDataset Class

Now we will create a custom dataset class to generate sequences and targets for training

Creating a custom `CharDataset` class is crucial because it prepares our text data into input sequences and target sequences that the RNN can learn from. By organizing the data this way, we can efficiently feed batches of sequences into the model during training, allowing it to learn the patterns of character sequences in the text.

In [78]:
# This is Cell #6

class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.vocab_size = vocab_size
        self.sequences = []
        self.targets = []
        
        # Create overlapping sequences with stride
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target
    

## Setting Hyperparameters

Now we will set our model's hyperparameters for our training process

Setting hyperparameters is important because they define the model's architecture and training behavior. They determine how the RNN processes data, learns patterns, and how quickly it converges during training. Properly chosen hyperparameters can significantly improve model performance and is a key step in training of models

Set the following hyperparameters for your model in the code cell below:
`sequence_length`, `stride`, `embedding_dim`, `hidden_size`, `num_layers`, `learning_rate`, `num_epochs`, `batch_size`, `vocab_size`.

In [185]:
# This is Cell #7

#TODO: Set your model's hyperparameters

sequence_length = 250  # Length of each input sequence
stride = 20            # Stride for creating sequences
embedding_dim = 128     # Dimension of character embeddings
hidden_size = 512    # Number of features in the hidden state of the RNN
learning_rate = 0.0005  # Learning rate for the optimizer
num_epochs = 5         # Number of epochs to train
batch_size = 64        # Batch size for training
vocab_size = len(vocab)
input_size = len(vocab)
output_size = len(vocab)


After you have set your hyperparameters in the code cell above, very breifly tell what is the role of each of the hyperparameter that you have defined above.
#### Hyperparameter Explanations & Justifications

After much experimentation and iteration with a wide variety of hyperparameters, I achieved my best loss and accuracy with the following configuration. 

1. **sequence_length = 250**:
   - A longer sequence length was chosen to capture more context from the text, as larger contexts are essential for understanding dependencies in a complex dataset such as `War and Peace`
   - **250** was chosen through much experimentation, where I found it balanced computational cost and the need for capturing meaningful patterns.

2. **stride = 20**:
   - A smaller stride ensures overlapping sequences, which helps the model learn continuity between characters. I increaded it slightly from the prior configuration, again through experimentation, but for the overall purpose of reducing excessive redundancy while maintaining sufficient data samples.

3. **embedding_dim = 128**:
   - I scaled this value with the increased sequence length and hidden side, as a higher embedded dimension size allows the model to distinguish more subtle nuances in the data.

4. **hidden_size = 512**:
   - Increasing the hidden size improves the model's ability to learn complex patterns and dependencies. For such a large vocabulary and text complexity, I found that **512** was a goof balance between the model's capacity and computational efficiency.

5. **learning_rate = 0.0005**:
   - A smaller learning rate stabilizes training for the more complex model, preventing overfitting during optimization. After a few iterations with a learning rate of **0.01**, I decided to lower it, and immediately saw steady improvements in the loss and accuracy over epochs.

6. **num_epochs = 5**:
   - I kept the number of epochs somewhat consistent with the original value of **3**. I experimented with higher epochs, such as **10**, and found that the model would converge in it's loss and accuracy around **5**. I found this value to be a sufficient number of opportunities for the model to learn.

7. **batch_size = 64**:
   - A batch size of **64** provides a good balance between gradient estimation accuracy and computational efficiency. Smaller batches would increase update frequency but at a computational cost, while larger batches might generalize less effectively for this dataset.

## Splitting Data into Training and Testing Sets

By now at this point in class, I'm confident that you know why we do this, so I'm not gonna say a lot here, let's jump right into the todo.

In [186]:
# This is Cell #8

data_tensor = torch.tensor(data, dtype=torch.long)

# Convert the data into a pytorch tensor and split the data into 90:10 ratio
train_size = int(0.9 * len(data_tensor))
train_data = data_tensor[:train_size]
test_data = data_tensor[train_size:]

# Print dimensions of tensors to verify split
print(f"Total data size: {len(data_tensor)}")
print(f"Training data size: {len(train_data)}")
print(f"Testing data size: {len(test_data)}")


Total data size: 3119966
Training data size: 2807969
Testing data size: 311997


## Creating Data Loaders

Now we will create data loaders for easy batching during training and testing.

Creating data loaders is essential to batch the data during training and testing. Batching allows the RNN to process multiple sequences in parallel, which speeds up training and makes better use of computational resources. 
We will also use Data loaders to shuffle the batched data, which is important for training models that generalize well.

Make sure to set `drop_last=True`

In [187]:
# This is Cell #9

train_dataset = CharDataset(train_data, sequence_length, stride, vocab_size)
test_dataset = CharDataset(test_data, sequence_length, stride, vocab_size)

# Initialize the training and testing data loader with batching and shuffling equal to True for training (and shuffling = False for testing)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

total_batches = len(train_loader)

# Output to confirm
print("Total batches in training data:", total_batches)

Total batches in training data: 2193


## Defining the RNN Model

Here we will define our character-level RNN model.

In [188]:
# This is Cell #10

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, embedding_dim=30):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = torch.nn.Embedding(output_size, embedding_dim)
        self.W_e = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.01)  # Smaller std
        self.b_e = nn.Parameter(torch.zeros(hidden_size))
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # Smaller std
        self.b_h = nn.Parameter(torch.zeros(hidden_size)) 
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        """
        x in [b, l] # b is batch_size and l is sequence length
        """
        x_embed = self.embedding(x)  # [b=batch_size, l=sequence_length, e=embedding_dim]
        b, l, _ = x_embed.size()
        x_embed = x_embed.transpose(0, 1) # [l, b, e]
        if hidden is None:
            h_t_minus_1 = self.init_hidden(b)
        else:
            h_t_minus_1 = hidden
        output = []
        for t in range(l):
            # RNN equation from the lecture 
            # We add a bias as well to expand the range of learnable functions
            h_t = torch.tanh(x_embed[t] @ self.W_e.T + self.b_e + h_t_minus_1 @ self.W_h.T + self.b_h) # [b, e]
            output.append(h_t)
            h_t_minus_1 = h_t
        output = torch.stack(output) # [l, b, e]
        output = output.transpose(0, 1) # [b, l, e]
        final_hidden = h_t.clone() # [b, h]
        logits = self.fc(output) # [b, l, vocab_size=v] 
        return logits, final_hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)


For a basic high level understanding of what is the CharRNN model that you just defined above, it consists of an embedding layer, an RNN layer, and a fully connected layer. Then embedding layer converts character indices into embeddings. Then RNN processes the embeddings and captures sequential information. Then finally the fully connected layer maps the RNN outputs to the vocabulary size for character prediction.


# Initializing the Model, Loss Function, and Optimizer

Now we will create an instance of the model that we just defined above and set up the loss function and optimizer. Then we will define a loss function, that evaluates the model's prediction against the true targets, and attaches a cost (number) on how good/bad the model is doing. During our training process, it is this cost that we try to minimize by tweaking the weights of the network. 

Then we will set up an optimizer, which will update the model's parameters based on the loss returned by the our loss function. This is how our model will learn over time.


In [189]:
# This is Cell #12

# Initialize your RNN model
model = CharRNN(input_size, hidden_size, output_size, embedding_dim).to(device)

# Define the loss function (use cross entropy loss)
criterion = nn.CrossEntropyLoss()

# Initialize your optimizer passing your model parameters and training hyperparameters
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


## Training the Model

Now finally, after all the setup that we have done, we can train our RNN. 

A basic idea high level idea of what we will do here is we will loop over epochs and batches to train the model. 
We will Initialize the hidden state at the beginning of each epoch. For each batch, we will reset the gradients, perform a forward pass, compute the loss, perform backpropagation, and update the model parameters. Then we detach the hidden state to prevent gradients from backpropagating through previous batches. We ill repeat this process for each batch. And finally we will calculate the average loss and accuracy for each epoch.
By performing forward and backward passes, calculating loss, and updating the model parameters, we enable the RNN to improve its predictions with each epoch.

In [190]:
# This is Cell #13

for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/5:   0%|          | 0/2193 [00:00<?, ?it/s]/var/folders/gr/clvj0jdn1vs8d33p_p7ws8lc0000gn/T/ipykernel_23961/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/var/folders/gr/clvj0jdn1vs8d33p_p7ws8lc0000gn/T/ipykernel_23961/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/5: 100%|██████████| 2193/2193 [15:52<00:00,  2.30it/s]


Epoch [1/5], Loss: 1.5837, Accuracy: 53.42%


Epoch 2/5: 100%|██████████| 2193/2193 [15:53<00:00,  2.30it/s]


Epoch [2/5], Loss: 1.3057, Accuracy: 61.07%


Epoch 3/5: 100%|██████████| 2193/2193 [15:50<00:00,  2.31it/s]


Epoch [3/5], Loss: 1.2491, Accuracy: 62.65%


Epoch 4/5: 100%|██████████| 2193/2193 [15:49<00:00,  2.31it/s]


Epoch [4/5], Loss: 1.2185, Accuracy: 63.51%


Epoch 5/5: 100%|██████████| 2193/2193 [15:54<00:00,  2.30it/s]

Epoch [5/5], Loss: 1.1980, Accuracy: 64.11%


## Check your loss

The training loss of your model when trained with a simple sequence like `"abcdefghijklmnopqrstuvwxyz" * 100` should be extremely close to zero. If that's not the case, go back and fix your bugs ;)

If you have acheived a training loss of 0 or extremley close to 0, then congratulations, lets move on to train your model with a bit more complicated sequence. That is our old favorite book, `warandpeace.txt`.

### Read the `warandpeace.txt` file

In [ ]:
# This is Cell #14

sequence = read_file('warandpeace.txt')

### Now Follow the instructions

1. Re-run Cell #5 to re-create character mappings for `warandpeace.txt`
2. Re-run Cell #7 to re-initialize hyperparameters
3. Re-run Cell #8 to split and create training and testing data with `warandpeace.txt` as your corpus
4. Re-run Cell #9 to set up data loaders with `warandpeace.txt` data
5. Re-run Cell #12 to re-initialize a new model object (maybe ask yourself why can't you use the previous model that was trained on the simple `"abc..."` corpus)
6. Re-run Cell #13 to train the new model with `warandpeace.txt` data.
   

## Evaluating the Model

After training, we evaluate the model on the test data.

In [191]:
# This is Cell #15

with torch.no_grad():
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in enumerate(test_loader):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten for CrossEntropyLoss

        total_loss += loss.item()

        # Calculate accuracy
        _, predicted_indices = torch.max(output, dim=2)  # Predicted characters
        correct_predictions += (predicted_indices == batch_targets).sum().item()
        total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

    avg_loss = total_loss / len(test_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    
    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

/var/folders/gr/clvj0jdn1vs8d33p_p7ws8lc0000gn/T/ipykernel_23961/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/var/folders/gr/clvj0jdn1vs8d33p_p7ws8lc0000gn/T/ipykernel_23961/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)


Test Loss: 1.3621, Test Accuracy: 59.70%


## Generating Text with the Trained Model

In this part of the assignment, your task is to implement the `generate_text` function, which uses a trained RNN model to generate text character-by-character, continuing from a given input. The function will produce an extended sequence by repeatedly predicting and appending the next character to the input.

### What the function is supposed to do?

1. Take an initial input text of length `n` from the user, convert it into indices using a predefined vocabulary (char_to_idx).
2. Use a trained model to predict the next character in the sequence.
3. Append the predicted character to the input, extend the input sequence, and repeat the process until `k` additional characters are generated.
4. Return the generated text, including the original input and the newly predicted characters.


In [193]:
# This is Cell #16

def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits (increase randomness with higher values)
    scaled_logits = logits / temperature  # Scale the logits by temperature
    # Apply softmax to convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)
    
    # Sample from the probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the probability distribution
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0):
    """
        model: The trained RNN model used for character prediction.
        start_text: The initial string of length `n` provided by the user to start the generation.
        n: The length of the initial input sequence.
        k: The number of additional characters to generate.
        temperature: Optional
        A scaling factor for randomness in predictions. Higher values (e.g., >1) make 
            predictions more random, while lower values (e.g., <1) make predictions more deterministic.
            Default is 1.0.
    """
    start_text = start_text.lower()
    #TODO: Implement the rest of the generate_text function
    
    # Prepare the initial input tensor
    input_sequence = torch.tensor([char_to_idx[char] for char in start_text], dtype=torch.long).unsqueeze(0).to(device)  # Shape: [1, n]
    
    # Initialize hidden state
    hidden = model.init_hidden(1)  # Batch size of 1

    generated_text = start_text  # Start with the initial text

    for _ in range(k):
        # Forward pass through the model
        output, hidden = model(input_sequence, hidden)

        # Get the latest output for the last character in the sequence
        last_output = output[:, -1, :]  # Shape: [1, vocab_size]
        
        # Sample the next character index
        next_char_index = sample_from_output(last_output, temperature).item()  # Get the index

        # Invalid indicies replaced with valid default index
        if next_char_index >= len(idx_to_char):
            next_char_index = 0

        # Append the predicted character to the generated text
        generated_text += idx_to_char[next_char_index]  # Convert index back to character

        # Update the input sequence with the new character
        input_sequence = torch.tensor([[next_char_index]], dtype=torch.long).to(device) # Update input sequence

    return generated_text

print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")
    
    if start_text.lower() == 'exit':
        print("Exiting...")
        break
    
    n = len(start_text) 
    k = int(input("Enter the number of characters to generate: "))
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0
    
    completed_text = generate_text(model, start_text, n, k, temperature)
    
    # Added Print Statements for Report
    print(f"\nStart Text: {start_text}")
    print(f"Number of Characters to Generate: {k}")
    print(f"Temperature: {temperature}")
    print(f"Generated text: {completed_text}")

Training complete. Now you can generate text.

Start Text: war and
Number of Characters to Generate: 20
Temperature: 0.1
Generated text: war and the same to see her

Start Text: war
Number of Characters to Generate: 30
Temperature: 0.5
Generated text: wars district that it was impossi

Start Text: hello how a
Number of Characters to Generate: 50
Temperature: 2.0
Generated text: hello how aida:whone kept vvnicalle, ringvnatol; no told tive

Start Text: hello how are
Number of Characters to Generate: 30
Temperature: 1.0
Generated text: hello how are you hopeless as she reticedib


ValueError: invalid literal for int() with base 10: ''

# Report Section

### Dataset 1: Alphabet Sample Data
**Sequence:** "abcdefghijklmnopqrstuvwxyz" * 100

**Final Training Metrics:**
- Epoch [5/5], Loss: 0.0003, Accuracy: 100.00%

##### **Performance Evaluation:**

For the training on the simpler dataset, the model achieved an impressively low loss and perfect accuracy, indicating that it learned the character sequence accurately.

#### **Generate Text Experiment:**
- **Start Text:** abc
  - **Number of Characters to Generate:** 5
  - **Temperature:** 0.1
  - **Generated text:** abcdefgh

- **Start Text:** abc
  - **Number of Characters to Generate:** 20
  - **Temperature:** 2.0
  - **Generated text:** abcmnouijklmnopqrstuvwx

- **Start Text:** abcdefg
  - **Number of Characters to Generate:** 12
  - **Temperature:** 1.0
  - **Generated text:** abcdefghijklmnopqrs

##### **Performance Evaluation:**

The model produced coherent and expected continuations when generating text based on the alphabet. 

### Dataset 2: Advanced Sample Data
**Sequence:** warandpeace.txt

**Final Training Metrics:**
- Epoch [5/5], Loss: 1.1980, Accuracy: 64.11%

**Final Test Metrics:**
  - Test Loss: 1.3621, Test Accuracy: 59.70%

##### **Performance Evaluation:**

Training on `warandpeace.txt`, the model achieved reasonable accuracy, but as expected, the loss values indicate that the model had more difficulty learning the complexities of this dataset compared to the simple sequence.

#### **Generate Text Experiment:**
- **Start Text:** war and
  - **Number of Characters to Generate:** 20
  - **Temperature:** 0.1
  - **Generated text:** war and the same to see her

- **Start Text:** war
  - **Number of Characters to Generate:** 30
  - **Temperature:** 0.5
  - **Generated text:** wars district that it was impossi

- **Start Text:** hello how a
  - **Number of Characters to Generate:** 50
  - **Temperature:** 2.0
  - **Generated text:** hello how aida:whone kept vvnicalle, ringvnatol; no told tive

- **Start Text:** hello how are
  - **Number of Characters to Generate:** 30
  - **Temperature:** 1.0
  - **Generated text:** hello how are you hopeless as she reticedib

##### **Performance Evaluation:**

The generated text from "warandpeace.txt" exhibited more complexity and varied outputs, although coherence sometimes suffered, especially with higher temperatures, leading to unpredictable word formations.

### Comparison Between Sequences

Overall, when comparing the two datasets, it's clear that the simpler dataset allowed the model to learn and generate text with high coherence and predictability. In contrast, training on `warandpeace.txt` revealed the challenges inherent in using a complex and nuanced dataset, where the output was considerably more erratic.

### Impact of Changing the Temperature Parameter

The temperature parameter significantly influenced the randomness of the text generated. Lower temperatures (e.g., 0.1) tightened the model's predictions, resulting in more coherent and logical outputs. Conversely, higher temperatures (e.g., 2.0) led to increased randomness, producing outputs that could be nonsensical or exhibit unusual character sequences.

### Overall Reflection

The coding process was relatively straightforward for the simpler dataset, but it became challenging when tuning hyperparameters for the more complex text. It required quite a bit of time to narrow in on an "optimal" configuration, especially due to the long training times. This assignment definitely emphasized the computational demands of training robust language models. The incoherence of generated text highlighted the limitations of the model, and gave me much insight into why large language models are often trained on days with incomprehendable datasets.
